<a href="https://colab.research.google.com/github/purnabandekar/TNS/blob/main/Decision_Trees_Random_Forests_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees and Random Forests — Practical

### Python Course · Colab Notebook

---

In the slides we drew a tree on paper. Here we let the computer build one, **open it up and
read it**, watch it go wrong when we let it grow too big, and then fix it with a forest.

The example is the same one from the slides: **will this student pass?**

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click a cell and press **Shift + Enter**.
2. **Every line of code has a comment** explaining what it does.
3. Most sections end with a **YOUR TURN** box — try it before moving on.
4. **Section 11 is a set of exercises with worked answers.**
5. Nothing to install, nothing to upload.

**Save your own copy first:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | The student data |
| 2 | Splitting into training and test |
| 3 | Growing the first tree |
| 4 | **Reading the tree** |
| 5 | Following one student through it |
| 6 | A tree draws boxes |
| 7 | Which question mattered most? |
| 8 | Letting the tree grow forever |
| 9 | The random forest |
| 10 | Scoring all three properly |
| 11 | **Exercises**, with answers |

---
## Section 1 — The student data

We need a table of students where we already know who passed, so the model has something to
learn from and we have something to check it against.

Rather than download a file, we will **make the data ourselves** in a few lines. That way
everybody's numbers are identical and nothing can go missing.

Each student has two facts:

- **hours_studied** — how many hours they studied,
- **attendance** — what percentage of classes they turned up to,

and one answer: **passed**, which is 1 for pass and 0 for fail.

In [ ]:
# import numpy, which we use to make the random numbers
import numpy as np
# import pandas, which holds the data as a table
import pandas as pd

# create a random number generator with a fixed seed
# the seed means "give me the same random numbers every time", so your table matches mine
rng = np.random.default_rng(15)

# how many students we want in our table
n_students = 400

# pick a study time between 0.5 and 10 hours for each student, rounded to 1 decimal place
hours_studied = np.round(rng.uniform(0.5, 10.0, n_students), 1)

# pick an attendance percentage between 40 and 100 for each student
# integers() stops one below the top number, so 101 gives us up to 100
attendance = rng.integers(40, 101, n_students)

Now the answer. We invent a sensible rule — studying and attending both help — and add some
randomness, because in real life two students with identical numbers do not always get the
same result.

In [ ]:
# build a hidden "readiness score" for each student out of the two facts
# studying counts for 6 points an hour, attendance for 0.9 points a percent
# the last part adds a random wobble, so the rule is not perfectly tidy
readiness = 6 * hours_studied + 0.9 * attendance + rng.normal(0, 8, n_students)

# a student passes (1) if their readiness is above 97, otherwise they fail (0)
passed = (readiness > 97).astype(int)

# now flip about 5% of the answers at random, to mimic real life
# rng.random() gives a number between 0 and 1 for each student, so "< 0.05" picks about 5%
surprises = rng.random(n_students) < 0.05
# np.where swaps 1 for 0 (and 0 for 1) wherever surprises is True, and leaves the rest alone
passed = np.where(surprises, 1 - passed, passed)

In [ ]:
# put the three columns together into one table
students = pd.DataFrame({"hours_studied": hours_studied,
                         "attendance": attendance,
                         "passed": passed})

# show the first 8 rows so we can see what a student looks like
print(students.head(8))

In [ ]:
# count how many students passed
print("passed:", int(np.sum(passed == 1)))
# count how many failed
print("failed:", int(np.sum(passed == 0)))
# print the total as a check
print("total :", len(students))

Roughly half and half, which is what we want — no lazy model can score well here just by
always saying the same thing.

> ### YOUR TURN
> Change the seed from `15` to any other number and re-run **from the cell containing the
> seed downwards** — the generator remembers where it got to, so re-running a later cell on
> its own gives different students even with the same seed.
>
> Do the pass and fail totals stay the same? **Put it back to 15 and re-run from the seed
> afterwards**, or every number below will differ from mine.

---
## Section 2 — Splitting into training and test

Same rule as always: the model learns from the **training** rows, and we judge it on the
**test** rows it has never seen. A tree is especially good at memorising, so this split is
what will catch it out later.

In [ ]:
# bring in the splitting function
from sklearn.model_selection import train_test_split

# X holds the facts we are allowed to use
X = students[["hours_studied", "attendance"]]
# y holds the answer we are trying to predict
y = students["passed"]

# keep 30% of the students hidden for testing, and fix the split with random_state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# how many students the tree may learn from
print("training students:", len(X_train))
# how many we kept back to score it
print("test students    :", len(X_test))

---
## Section 3 — Growing the first tree

`max_depth=2` means **"you may ask at most two questions"**. That keeps the tree small
enough to print on one screen, which is the whole point of this session.

In [ ]:
# bring in the decision tree
from sklearn.tree import DecisionTreeClassifier

# create the tree, allowing it to ask at most 2 questions before deciding
# random_state=0 makes it break any ties the same way every time
small_tree = DecisionTreeClassifier(max_depth=2, random_state=0)

# grow the tree from the training students
small_tree.fit(X_train, y_train)

# print a message so we know it finished
print("tree grown from", len(X_train), "students")

In [ ]:
# bring in the accuracy score from the last session
from sklearn.metrics import accuracy_score

# how often it is right on the students it learnt from
train_accuracy = accuracy_score(y_train, small_tree.predict(X_train))
# how often it is right on the students it has never seen
test_accuracy = accuracy_score(y_test, small_tree.predict(X_test))

# print the training score
print("accuracy on training students:", round(train_accuracy, 4))
# print the test score, which is the one that counts
print("accuracy on test students    :", round(test_accuracy, 4))

About 83% on students it has never seen — from **two questions**. Hold on to that number;
we will beat it later, and we will also destroy it.

---
## Section 4 — Reading the tree

This is the thing decision trees do that almost no other model does: **you can look
inside**.

In [ ]:
# bring in the function that draws a tree as a picture
from sklearn.tree import plot_tree
# bring in matplotlib, the plotting library
import matplotlib.pyplot as plt

# set the size of the picture in inches
plt.figure(figsize=(11, 6))
# draw the tree
# filled=True colours each box by the answer, feature_names puts our column names on it,
# class_names labels the two answers, rounded just makes the boxes look nicer
plot_tree(small_tree, filled=True, rounded=True,
          feature_names=["hours_studied", "attendance"],
          class_names=["fail", "pass"])
# show the picture
plt.show()

The same tree also prints as plain text, which is easier to read out loud.

In [ ]:
# bring in the text version
from sklearn.tree import export_text

# print the tree as indented text, using our column names
print(export_text(small_tree, feature_names=["hours_studied", "attendance"]))

### What it is telling us

Read it as nested questions:

- **Did they study more than 6.25 hours?**
  - **No** → then they only pass if **attendance is above 78.5%**.
  - **Yes** → then they pass as long as **attendance is above 60.5%**.

That is a genuinely sensible rule, and the computer worked it out on its own: *the less you
study, the more you have to turn up.* Nobody programmed that in.

**Two words from the slides, now visible in the picture:**

- **`samples`** — how many training students landed in that box.
- **`gini`** — how mixed that box is. **0 means perfectly pure** (all one answer), and the
  closer to 0.5, the more mixed. The tree picks whichever question drops gini the most.

> ### YOUR TURN
> Find the leaf with the lowest `gini` in the picture. How many students are in it, and
> what does a low gini tell you about them?

---
## Section 5 — Following one student through it

Let us take a student who studied only 3 hours but attended 88% of classes, and walk the
tree by hand before asking the computer.

- Studied more than 6.25 hours? **No** (3 hours).
- Attendance above 78.5%? **Yes** (88%).
- → the tree should say **pass**.

In [ ]:
# build a one-row table for our student, with the same column names as the training data
one_student = pd.DataFrame([[3.0, 88]], columns=["hours_studied", "attendance"])

# ask the tree for its answer, [0] pulls the single answer out of the list it returns
answer = small_tree.predict(one_student)[0]

# print the raw answer, which is 1 for pass and 0 for fail
print("the tree says:", answer)
# turn it into a word, so it reads properly
if answer == 1:
    # 1 means the tree predicted a pass
    print("in words     : PASS")
# otherwise the answer was 0
else:
    # 0 means the tree predicted a fail
    print("in words     : FAIL")

No maths, no formula — we followed two questions on paper and got the same answer the
computer did. **That is what people mean when they call a model "explainable".**

> ### YOUR TURN
> Change the student to `[[7.0, 55]]` and work out the answer on paper first: studied more
> than 6.25 hours (yes), attendance above 60.5% (no). Run it and check.
>
> **Then put it back to `[[3.0, 88]]` and re-run the cell**, because Section 9 uses this
> same student again.

---
## Section 6 — A tree draws boxes

A tree can only ever ask questions like *"is this number above that number?"*, so the
picture it draws is always made of **rectangles with straight edges**, never a slope or a
curve.

Let us colour in what the tree predicts for every possible student, and drop the real
students on top.

In [ ]:
# bring in the helper that colours in a model's answer across a whole picture
from sklearn.inspection import DecisionBoundaryDisplay

# make one picture, 6 by 5 inches, and keep a handle on it called "ax"
figure, ax = plt.subplots(figsize=(6, 5))

# colour every point of the background with whatever the tree would answer there
# response_method="predict" asks for the plain pass/fail answer, so we get solid blocks
# alpha=0.3 makes the colours pale, so we can still see the dots on top
DecisionBoundaryDisplay.from_estimator(small_tree, X_train, ax=ax, alpha=0.3,
                                       cmap="coolwarm", response_method="predict")

# now scatter the real training students on top
# c=y_train colours each dot by whether that student really passed
# edgecolor="k" draws a thin black outline so the dots stand out
ax.scatter(X_train["hours_studied"], X_train["attendance"],
           c=y_train, cmap="coolwarm", edgecolor="k", s=25)

# label the bottom axis
ax.set_xlabel("hours studied")
# label the side axis
ax.set_ylabel("attendance (%)")
# give the picture a title
ax.set_title("What the small tree predicts everywhere")
# show it
plt.show()

**In these pictures red means the tree says pass and blue means it says fail**, and the dots
are coloured the same way for what really happened. (Careful: the tree diagram in Section 4
used sklearn's own colours, where blue meant *pass*. The two pictures are not related.)

Look at the edges between the colours: they are **straight, and they meet at right angles**.
Every boundary is one of the three questions we read in Section 4 — a vertical cut at 6.25
hours, and horizontal cuts at 78.5% and 60.5%.

A straight-line model like logistic regression would have split this picture with one
sloped line running all the way across. A tree draws steps instead. Neither shape is better
everywhere — but the tree's version is the one you can read out loud.

> ### YOUR TURN
> Find the vertical line in the picture. Which of the two questions is it, and why is that
> one vertical rather than horizontal?

---
## Section 7 — Which question mattered most?

The tree can also tell us **how much work each column did**. Bigger number = that column
was used for splits that tidied up the data more.

In [ ]:
# the model stores one importance number per column, in the same order as the columns
importances = small_tree.feature_importances_

# print the importance of hours studied, rounded so it is readable
print("hours_studied:", round(importances[0], 3))
# print the importance of attendance
print("attendance   :", round(importances[1], 3))
# the numbers always add up to 1, because they are shares of the total work
print("they add up to:", round(importances[0] + importances[1], 3))

The two come out close to an even split — but look at the tree again before you decide why.
`hours_studied` was used **once**, right at the top, where that single question had to sort
all 280 students at once. `attendance` was used **twice**, lower down, on two smaller
groups. One big question and two small ones happen to be worth about the same here.

**A warning worth remembering:** importance says *how useful a column was to this tree*.
It does **not** prove that studying causes passing, and a column can score low simply
because another column already captured the same information.

> ### YOUR TURN
> Which column would you expect to score higher on a tree of depth 1, which only gets to
> ask **one** question? Try it: build `DecisionTreeClassifier(max_depth=1)`, fit it, and
> print its importances.

---
## Section 8 — Letting the tree grow forever

Now the failure mode from the slides. We remove `max_depth` entirely, so the tree keeps
asking questions for as long as it can make a box any tidier — which on this data means
until every single training student is sorted correctly.

In [ ]:
# create a tree with no depth limit at all, so it can grow as far as it likes
full_tree = DecisionTreeClassifier(random_state=0)
# grow it on exactly the same training students
full_tree.fit(X_train, y_train)

# how many end boxes (leaves) it ended up with
print("leaves in the small tree:", small_tree.get_n_leaves())
# the unlimited tree will have far more
print("leaves in the full tree :", full_tree.get_n_leaves())
# and how many questions deep it goes
print("depth of the full tree  :", full_tree.get_depth())

In [ ]:
# score the full tree on the students it learnt from
full_train = accuracy_score(y_train, full_tree.predict(X_train))
# score it on the students it has never seen
full_test = accuracy_score(y_test, full_tree.predict(X_test))

# print the full tree's two scores
print("FULL tree  -> training:", round(full_train, 4), "  test:", round(full_test, 4))
# print the small tree's two scores underneath, for comparison
print("SMALL tree -> training:", round(train_accuracy, 4), "  test:", round(test_accuracy, 4))

### Read those two lines carefully

The full tree got **every single training student right** — a perfect 1.0. And on new
students it is **worse than the two-question tree**.

It did not learn the rule. It learnt the 280 individual students, including the ~5% whose
answers we flipped at random. Those flipped students have nothing to teach anybody, and the
tree carefully memorised them anyway.

**That is overfitting**, and the training score is exactly where it hides: a model that
looks perfect on the data it has seen is not a good model, it is a suspicious one.

Here is what that memorising looks like.

In [ ]:
# make one row of two pictures side by side, 12 by 5 inches in total
figure, axes = plt.subplots(1, 2, figsize=(12, 5))

# colour in what the SMALL tree predicts, on the left-hand picture
DecisionBoundaryDisplay.from_estimator(small_tree, X_train, ax=axes[0], alpha=0.3,
                                       cmap="coolwarm", response_method="predict")
# scatter the training students on top of it
axes[0].scatter(X_train["hours_studied"], X_train["attendance"],
                c=y_train, cmap="coolwarm", edgecolor="k", s=20)
# title the left-hand picture
axes[0].set_title("small tree (2 questions)")

# colour in what the FULL tree predicts, on the right-hand picture
DecisionBoundaryDisplay.from_estimator(full_tree, X_train, ax=axes[1], alpha=0.3,
                                       cmap="coolwarm", response_method="predict")
# scatter the same students on top of it
axes[1].scatter(X_train["hours_studied"], X_train["attendance"],
                c=y_train, cmap="coolwarm", edgecolor="k", s=20)
# title the right-hand picture
axes[1].set_title("full tree (no limit)")

# show both pictures
plt.show()

The right-hand picture is covered in thin slivers and single-dot islands. Each one is the
tree bending itself around one awkward student. None of them will help with the next
student who walks in.

### So how deep *should* it be?

Try several depths and watch the two scores separate.

In [ ]:
# a list of depths we want to try
depths = [1, 2, 3, 4, 5, 6, 8, 10]
# an empty list to collect the training score at each depth
train_scores = []
# an empty list to collect the test score at each depth
test_scores = []

# go through the depths one at a time
for d in depths:
    # build a tree allowed to ask d questions
    t = DecisionTreeClassifier(max_depth=d, random_state=0)
    # grow it on the training students
    t.fit(X_train, y_train)
    # record how it did on the training students
    train_scores.append(accuracy_score(y_train, t.predict(X_train)))
    # record how it did on the test students
    test_scores.append(accuracy_score(y_test, t.predict(X_test)))
    # print the pair as we go
    print("depth", d, "-> training:", round(train_scores[-1], 3),
          "  test:", round(test_scores[-1], 3))

In [ ]:
# set the picture size
plt.figure(figsize=(7, 4.5))
# draw the training scores as a line with round markers
plt.plot(depths, train_scores, marker="o", label="training accuracy")
# draw the test scores as a second line with square markers
plt.plot(depths, test_scores, marker="s", label="test accuracy")
# label the bottom axis
plt.xlabel("max_depth (how many questions the tree may ask)")
# label the side axis
plt.ylabel("accuracy")
# title the picture
plt.title("The deeper it goes, the more it memorises")
# show which line is which
plt.legend()
# show the picture
plt.show()

The training line climbs steadily until it reaches a perfect 1.0. The test line jumps up
early, then flattens off and wobbles — best at depth 3, and clearly worse by depth 10.

Do not read anything into the small wobbles: with only 120 test students, one student
changing sides moves that line by almost a whole percent. What *does* matter is that the two
lines keep drifting apart. **The gap between them is the memorising.**

> ### YOUR TURN
> Read the printed numbers and find the depth with the best test score. Is it the deepest
> tree? Would you have picked it by looking at the training scores alone?

---
## Section 9 — The random forest

One tree left unsupervised memorises. The fix is not a cleverer tree — it is **lots of
them**.

A random forest grows many trees, each on a slightly different random sample of the
students and allowed to consider only some of the columns at each question. So every tree
is a bit different, and each one memorises **different** awkward students. Then they vote,
and the odd mistakes cancel out.

`n_estimators=100` means one hundred trees.

In [ ]:
# bring in the random forest
from sklearn.ensemble import RandomForestClassifier

# create a forest of 100 trees, with a fixed random_state so your result matches mine
forest = RandomForestClassifier(n_estimators=100, random_state=0)
# grow all 100 trees on the same training students
forest.fit(X_train, y_train)

# score the forest on the training students
forest_train = accuracy_score(y_train, forest.predict(X_train))
# score the forest on the test students
forest_test = accuracy_score(y_test, forest.predict(X_test))

# print the two-question tree's scores
print("SMALL tree -> training:", round(train_accuracy, 4), "  test:", round(test_accuracy, 4))
# print the unlimited tree's scores
print("FULL tree  -> training:", round(full_train, 4), "  test:", round(full_test, 4))
# print the forest's scores
print("FOREST     -> training:", round(forest_train, 4), "  test:", round(forest_test, 4))

Notice the forest's **training** score is 1.0, exactly like the full tree — every tree
inside it is still an unlimited tree, and they still memorise.

But on students it has never seen it is the best of the three. **The memorising did not go
away; it got voted out.**

In [ ]:
# ask the forest about our 3-hour, 88% student again
forest_answer = forest.predict(one_student)[0]

# count how many of the trees voted "pass" for this student
votes_for_pass = 0
# go through every tree inside the forest
for tree in forest.estimators_:
    # .to_numpy() hands over just the numbers, which is what each inner tree expects
    # [0] pulls out that single tree's answer for our one student
    if tree.predict(one_student.to_numpy())[0] == 1:
        # add one to the count if this tree voted pass
        votes_for_pass = votes_for_pass + 1

# print how the vote went
print("trees voting PASS:", votes_for_pass, "out of", len(forest.estimators_))
# print the forest's final answer, which is simply the majority
print("the forest says  :", forest_answer)

That is the whole trick: **count the votes, take the majority.**

Now the point that makes forests worth using. Score each of the 100 trees **on its own**,
and compare them with the forest they add up to.

In [ ]:
# a list to collect each individual tree's test accuracy
single_scores = []
# go through every tree inside the forest
for tree in forest.estimators_:
    # score this one tree on its own, on the test students
    single_scores.append(accuracy_score(y_test, tree.predict(X_test.to_numpy())))

# the worst single tree
print("worst single tree :", round(min(single_scores), 4))
# the average single tree
print("average single tree:", round(float(np.mean(single_scores)), 4))
# the best single tree
print("best single tree  :", round(max(single_scores), 4))
# the forest, which is those same trees voting together
print("the forest        :", round(forest_test, 4))

In [ ]:
# count how many of the 100 trees actually beat the forest on their own
better_than_forest = 0
# go through every score we collected
for s in single_scores:
    # compare this tree's score with the forest's score
    if s > forest_test:
        # count it if this tree did better on its own
        better_than_forest = better_than_forest + 1

# print the count
print("single trees that beat the whole forest:", better_than_forest)

The average tree in there is worse than the two-question tree we started with, and the worst
one is worse than the unlimited tree we just spent a whole section condemning. One single tree happens to **match** the forest, and **not one beats it.**

So a hundred so-so opinions, averaged, come out as good as the single best opinion that went
into them — and here is the part that matters: **you had no way of knowing in advance which
tree the lucky one would be.** The forest gets you that score without having to guess.

> ### YOUR TURN
> Change `n_estimators=100` to `n_estimators=5` and re-run. Does the test score get worse?
> Now try `n_estimators=500` — does it keep improving forever, or does it flatten out?
>
> **Then put it back to `n_estimators=100` and re-run the cell**, or every number from here
> to the end will differ from mine.

---
## Section 10 — Scoring all three properly

Accuracy alone is not enough, as we saw last session. Here are all three models on all four
metrics, on the test students.

For this data the positive class (`1`) is **pass**, so:

- **precision** — of the students it predicted would pass, how many really did?
- **recall** — of the students who really passed, how many did it spot?

In [ ]:
# bring in the other three metrics
from sklearn.metrics import precision_score, recall_score, f1_score

# make a small function so we do not repeat the same four lines three times
def score_model(name, predictions):
    # print the model's name on its own line, with its four scores indented underneath
    print(name)
    # accuracy: how often it was right overall
    print("   accuracy :", round(accuracy_score(y_test, predictions), 4))
    # precision: when it said pass, how often was it right
    print("   precision:", round(precision_score(y_test, predictions), 4))
    # recall: of the real passes, how many it found
    print("   recall   :", round(recall_score(y_test, predictions), 4))
    # F1: the single number balancing precision and recall
    print("   F1       :", round(f1_score(y_test, predictions), 4))

# score the two-question tree
score_model("SMALL tree", small_tree.predict(X_test))
# score the unlimited tree
score_model("FULL tree", full_tree.predict(X_test))
# score the forest
score_model("FOREST", forest.predict(X_test))

In [ ]:
# bring in the confusion matrix so we can see the forest's actual mistakes
from sklearn.metrics import confusion_matrix

# build the matrix for the forest, true answers first
forest_cm = confusion_matrix(y_test, forest.predict(X_test))

# print the square: rows are the truth, columns are the prediction
print(forest_cm)
# spell out the false alarms, which sit in the top-right cell
print("students who failed but it said pass:", forest_cm[0][1])
# spell out the missed passes, which sit in the bottom-left cell
print("students who passed but it said fail:", forest_cm[1][0])

The forest wins on every one of the four, and the full tree loses on every one of the four.
The ranking is the same whichever metric you pick — which is a comfortable position to be
in, and not one you should expect every time.

> ### YOUR TURN
> Suppose this model were used to decide who gets extra tutoring. Which mistake would you
> rather the model made — telling a struggling student they will pass, or sending someone
> to tutoring who did not need it? Which metric would you watch?

---
## Section 11 — Exercises

Try each one before opening the answer underneath it.

### Exercise 1 — find the best depth

Section 8 tried eight different depths, from 1 up to 10. Build the tree at the depth that
scored best on the test students, and check whether it also beats the forest.

In [ ]:
# build a tree at the best depth we found in Section 8
best_tree = DecisionTreeClassifier(max_depth=3, random_state=0)
# grow it on the training students
best_tree.fit(X_train, y_train)

# score it on the test students
print("depth-3 tree test accuracy:", round(accuracy_score(y_test, best_tree.predict(X_test)), 4))
# print the forest's score next to it
print("forest test accuracy      :", round(forest_test, 4))

**Answer.** Depth 3 was the best of the depths we tried, at about 0.85 — better than both
the two-question tree and the unlimited tree. It still loses to the forest.

**One honest warning.** We chose depth 3 by looking at the *test* scores, so that 0.85 is a
slightly flattering number: we tried eight depths on the same 120 students and kept the
luckiest one. Proper practice is to hold back a **third** slice of data for choosing
settings, and to touch the test set only once, right at the end. We have cut a corner here
to keep the notebook short — but know that it is a corner.

The lesson is not "depth 3 is the right answer". It is that the right depth has to be
**measured on data the tree did not learn from**, and that even the best single tree here
was beaten by a crowd of worse ones.

### Exercise 2 — a forest of stumps

What happens if you build a forest out of deliberately terrible trees — ones allowed only a
single question each?

In [ ]:
# a forest of 100 trees, each allowed only ONE question (a "stump")
stump_forest = RandomForestClassifier(n_estimators=100, max_depth=1, random_state=0)
# grow it on the training students
stump_forest.fit(X_train, y_train)

# score the stump forest on the test students
print("forest of stumps :", round(accuracy_score(y_test, stump_forest.predict(X_test)), 4))
# print the normal forest for comparison
print("normal forest    :", round(forest_test, 4))
# and the single two-question tree
print("single small tree:", round(test_accuracy, 4))

**Answer.** The stump forest scores about 0.80 — clearly worse than the normal forest, and
worse even than the single two-question tree.

Voting rescues trees that are *individually noisy but roughly right*; it cannot rescue
trees that are all too simple to see the pattern in the first place. A hundred people who
each only looked at one number still all miss the same thing.

### Exercise 3 — read a new tree out loud

Build a tree of depth 2 on **attendance only**, print it, and say its rule in one English
sentence.

In [ ]:
# keep just the attendance column, the double brackets keep it as a table
X_train_one = X_train[["attendance"]]
# do the same for the test students
X_test_one = X_test[["attendance"]]

# build a depth-2 tree that can only look at attendance
attendance_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
# grow it on the training students
attendance_tree.fit(X_train_one, y_train)

# print the tree as text
print(export_text(attendance_tree, feature_names=["attendance"]))
# score it on the test students
print("test accuracy:", round(accuracy_score(y_test, attendance_tree.predict(X_test_one)), 4))

**Answer.** In one sentence: *attendance above 59.5% → pass, otherwise fail.*

Look closely at what it printed. It was allowed two questions and it used them — but on
both sides, **the two leaves give the same answer**.

The extra questions were not useless to the tree: each one made its boxes a little tidier
(a little lower in gini), and that is the only thing sklearn needs in order to split. But
neither was enough to flip a whole leaf from fail to pass. So all that extra work is
invisible in the predictions, and a depth-2 tree behaves exactly like a single rule.

It scores about 0.76, well below the 0.83 we got by also looking at study hours. **Taking a
column away costs you accuracy** — which is the honest version of what feature importance
was hinting at in Section 7.

### Exercise 4 — how many trees are enough?

Score forests of 1, 5, 20, 100 and 300 trees. Where does the improvement stop being worth
the wait?

In [ ]:
# the forest sizes we want to try
sizes = [1, 5, 20, 100, 300]

# try each size in turn
for k in sizes:
    # build a forest of k trees
    f = RandomForestClassifier(n_estimators=k, random_state=0)
    # grow it on the training students
    f.fit(X_train, y_train)
    # print the size and its test accuracy
    print("trees:", k, "-> test accuracy:", round(accuracy_score(y_test, f.predict(X_test)), 4))

**Answer.** The scores barely move: 1, 5 and 20 trees all land on the same number here, 100
nudges it up, and 300 is no better than 100 — just three times slower.

Two things worth taking from that. First, **more trees is not how you make a forest good** —
past a certain size the votes stop changing, and the extra trees only cost you time. Adding
trees will not make it memorise the way extra depth did, so a large number is safe; it is
just pointless.

Second, do not read much into any one of these numbers. They are all measured on the same
120 test students, so **one student changing sides moves the score by almost a whole
percent** — and the "forest" of 1 tree is a single random tree that simply got lucky with
this seed.

### Exercise 5 — the summary question

In two or three sentences, explain to somebody who missed the session why we would use a
random forest instead of a single decision tree — and what we give up by doing so.

**Answer.** A good answer says something like: *a single tree left to grow memorises the
training data and does worse on new students, while a forest of a hundred different trees
votes and cancels those individual mistakes out — so it predicts better. What we give up is
the picture: we could read the two-question tree out loud, but nobody can read a hundred
trees, so the forest is far harder to explain to the person it makes a decision about.*

Naming that trade — **accuracy up, explainability down** — is the mark of a good answer.

---
## Recap

| | what it is | strength | weakness |
|---|---|---|---|
| **Decision tree** | a chain of yes/no questions | you can read it and explain it | left unlimited, it memorises |
| **Random forest** | many different trees voting | much steadier on new data | you cannot read it any more |

**Three things worth keeping:**

1. A perfect **training** score is a warning sign, not an achievement.
2. `max_depth` is the setting that keeps a single tree honest — and the right value is
   measured on data you held back for that purpose, not guessed.
3. A forest fixes the memorising **by averaging it away**, not by making any single tree
   smarter.